In [ ]:
# ==========================================
# Multiclass Classification Challenge (10C-W26)
# Author: Martin Sure
# ==========================================
# This solution uses:
# - Custom CNN (ResNet-style) built from scratch
# - Stratified K-Fold Cross Validation
# - MixUp Data Augmentation
# - Model Ensembling
# - Cosine Learning Rate Scheduler
#
# NOTE:
# No pre-trained models were used in this solution.
# ==========================================

# -------------------------
# IMPORTS
# -------------------------
import os
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from google.colab import drive
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

In [ ]:
# -------------------------
# CONFIGURATION
# -------------------------
NUM_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 25
NUM_FOLDS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Mount Drive (adjust base_path as needed)
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/kaggle challange'  # Your path to pkl files

Mounted at /content/drive


In [ ]:
# -------------------------
# LOAD DATA
# -------------------------
with open(os.path.join(base_path, 'train_X_y.pkl'), 'rb') as f:
    X_train_full, y_train_full = pickle.load(f)

with open(os.path.join(base_path, 'test_X.pkl'), 'rb') as f:
    X_test = pickle.load(f)

In [ ]:
# Normalize
X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
y_train_full = y_train_full.flatten()

In [ ]:
# -------------------------
# DATASET CLASS
# -------------------------
class ImageDataset(Dataset):
    def __init__(self, images, labels=None, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx].transpose(2, 0, 1)  # HWC → CHW
        img = torch.tensor(img, dtype=torch.float32)

        if self.transform:
            img = self.transform(img)

        if self.labels is not None:
            return img, self.labels[idx]
        return img


In [ ]:
# -------------------------
# TRANSFORMS
# -------------------------
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2),
])

test_transform = transforms.Compose([])

In [ ]:
# -------------------------
# MIXUP
# -------------------------
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(DEVICE)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

In [ ]:
# -------------------------
# MODEL: CUSTOM RESNET
# -------------------------
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(out_c)

        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(out_c)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride),
                nn.BatchNorm2d(out_c)
            )

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return torch.relu(out)

class CustomResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(3, 64, 3, padding=1)
        self.bn = nn.BatchNorm2d(64)

        self.layer1 = self._make_layer(64, 64, 2, 1)
        self.layer2 = self._make_layer(64, 128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)

        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(256, NUM_CLASSES)

    def _make_layer(self, in_c, out_c, blocks, stride):
        layers = [ResidualBlock(in_c, out_c, stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_c, out_c))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = torch.relu(self.bn(self.conv(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [ ]:
skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

fold_models = []
val_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):

    print(f"\n===== Fold {fold+1} =====")

    checkpoint_path = f"{base_path}/fold_{fold}_checkpoint.pth"
    best_model_path = f"{base_path}/fold_{fold}_best.pth"

    train_dataset = ImageDataset(X_train_full[train_idx], y_train_full[train_idx], train_transform)
    val_dataset = ImageDataset(X_train_full[val_idx], y_train_full[val_idx], test_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = CustomResNet().to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    start_epoch = 0
    best_acc = 0


===== Fold 1 =====

===== Fold 2 =====

===== Fold 3 =====

===== Fold 4 =====

===== Fold 5 =====


In [ ]:
if os.path.exists(checkpoint_path):
    print("Resuming from checkpoint...")
    start_epoch, best_acc = load_checkpoint(checkpoint_path, model)

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        train_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            images, y_a, y_b, lam = mixup_data(images, labels)

            optimizer.zero_grad()
            outputs = model(images)

            loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        scheduler.step()

        # --- Validation ---
        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                correct += (outputs.argmax(1) == labels).sum().item()

        val_acc = correct / len(val_dataset)

        # ✅ Print inside the loop
        print(f"Fold {fold+1} | Epoch {epoch+1}: Val Acc = {val_acc:.4f}")

In [ ]:
# -------------------------
# VALIDATION SUMMARY
# -------------------------
print("\nValidation Accuracies per Fold:", val_accuracies)
print("Mean Validation Accuracy:", np.mean(val_accuracies))


Validation Accuracies per Fold: []
Mean Validation Accuracy: nan


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [ ]:
# -------------------------
# ENSEMBLE PREDICTION
# -------------------------
test_dataset = ImageDataset(X_test, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

test_preds = np.zeros((len(X_test), NUM_CLASSES))

for model in fold_models:
    model.eval()
    preds = []

    with torch.no_grad():
        for images in test_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds.append(outputs.softmax(dim=1).cpu().numpy())

    test_preds += np.concatenate(preds)

test_preds /= NUM_FOLDS
final_predictions = np.argmax(test_preds, axis=1)

In [ ]:
# -------------------------
# SAVE SUBMISSION
# -------------------------
submission = pd.DataFrame({
    'rowId': range(len(final_predictions)),
    'target': final_predictions
})

submission.to_csv(os.path.join(base_path, '3rd_submission.csv'), index=False)

print("\n Submission file saved successfully!")
print(submission.head())


 Submission file saved successfully!
   rowId  target
0      0       0
1      1       0
2      2       0
3      3       0
4      4       0
